# Last.fm latest: baseline và T-KGAT trên Kaggle

Dùng bộ **25% user, seed 2019, lần nghe muộn nhất** đã dựng trên máy. Gói Input phải là **Private**. Chạy code PyTorch trong `src/`; không đọc lại Drive.

Đủ 5 seed 2019–2023; Stage 1 và Stage 2 quét riêng **γ ∈ {1, 3, 10}**, chọn theo **Recall@20 trung bình trên validation**, rồi mới chấm test cho cấu hình được chọn. Không chọn theo test.

Mặc định chỉ chuẩn bị môi trường. Chốt thí nghiệm với team trước khi bật `RUN_TRAINING`. Xem `docs/lastfm-kaggle.md` trong gói để biết cách chia phiên và khôi phục.

In [ ]:
from pathlib import Path

# Set these paths only when automatic discovery finds multiple inputs.
PAYLOAD_DIR = None
RESTORE_ZIP = None
PROJECT_ROOT = Path('/kaggle/working/t-kgat')

RUN_MEASUREMENT = False
RUN_TRAINING = False
RUN_REPORT = False
STEPS_THIS_SESSION = 1
NEW_KG_FITS_PER_STEP = 2


## Chuẩn bị code và dữ liệu

Thêm Dataset chứa `tkgat-lastfm-latest-kaggle.zip` qua Add Input. Để tiếp tục phiên trước, thêm output có `tkgat-lastfm-results.zip` vào Input. Ô dưới kiểm tra hash và clone phiên bản source đã đóng gói; chưa train.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import PurePosixPath
from zipfile import ZipFile

input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
if PAYLOAD_DIR is None:
    candidates = sorted(input_root.rglob('payload-manifest.json'))
    if not candidates:
        archives = sorted(input_root.rglob('tkgat-lastfm-latest-kaggle.zip'))
        if len(archives) != 1:
            raise RuntimeError('Add the private payload Dataset, or set PAYLOAD_DIR explicitly.')
        unpacked = working / 'tkgat-payload'
        with ZipFile(archives[0]) as archive:
            for name in archive.namelist():
                p = PurePosixPath(name)
                if p.is_absolute() or '..' in p.parts or '\\' in name:
                    raise ValueError(f'Invalid ZIP member: {name}')
            archive.extractall(unpacked)
        candidates = [unpacked / 'payload-manifest.json']
    identities = {p.read_text() for p in candidates}
    if len(identities) != 1:
        raise RuntimeError('Multiple payloads found. Set PAYLOAD_DIR to the intended Dataset directory.')
    payload_root = candidates[0].parent
else:
    payload_root = Path(PAYLOAD_DIR)
manifest = json.loads((payload_root / 'payload-manifest.json').read_text())
bundle = payload_root / 'source.bundle'
digest = hashlib.sha256()
with bundle.open('rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        digest.update(chunk)
if digest.hexdigest() != manifest['files_sha256']['source.bundle']:
    raise ValueError('Source bundle hash mismatch')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', str(bundle), str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if commit != manifest['source_commit']:
    raise RuntimeError('Existing working copy belongs to another payload. Start a fresh session.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
args = [sys.executable, '-u', 'scripts/kaggle_lastfm.py', 'prepare', '--payload', str(payload_root)]
has_progress = (PROJECT_ROOT / 'results/registry.csv').exists() or any((PROJECT_ROOT / 'results/runs').rglob('*.pt'))
if not has_progress:
    restores = [Path(RESTORE_ZIP)] if RESTORE_ZIP else sorted(input_root.rglob('tkgat-lastfm-results.zip'))
    if len(restores) > 1:
        raise RuntimeError('Multiple checkpoint archives found. Set RESTORE_ZIP explicitly.')
    if restores:
        args += ['--restore', str(restores[0])]
subprocess.run(args, check=True)
print('Working copy:', PROJECT_ROOT)
print('Export snapshot:', commit)


## Kiểm tra GPU và đo chi phí

GPU phải hiện `CUDA: True`. Chỉ bật `RUN_MEASUREMENT = True` khi muốn đo vài epoch train/val; bước đo không mở test và không ghi điểm vào registry.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU used:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a GPU in Kaggle Accelerator before training.')
if RUN_MEASUREMENT:
    subprocess.run([sys.executable, '-u', 'scripts/kaggle_lastfm.py', 'measure'], check=True)
else:
    print('Measurement disabled. Set RUN_MEASUREMENT = True when ready.')


## Chạy lần lượt bộ thí nghiệm

Bật `RUN_TRAINING = True` sau khi chốt. Thứ tự: MostPop → BPR-MF → KGAT → KGAT không KG → Stage 1 → đối chứng Stage 1 → Stage 2 → đối chứng Stage 2.

Mỗi bước KGAT/T-KGAT chỉ làm tối đa `NEW_KG_FITS_PER_STEP` lượt mới. **Vẫn chạy đủ toàn bộ lưới và 5 seed** qua các bước tiếp theo; chỉ chọn cấu hình/chấm test sau khi đủ lưới. MostPop và BPR-MF chạy trọn mô hình; BPR-MF là 15 lượt nên dành riêng một phiên cho bước này. Dựa vào thời gian đo để tăng số bước vừa với phiên của bạn.

In [ ]:
if RUN_TRAINING:
    subprocess.run([
        sys.executable, '-u', 'scripts/kaggle_lastfm.py', 'train',
        '--steps', str(STEPS_THIS_SESSION), '--max-new-runs', str(NEW_KG_FITS_PER_STEP),
    ], check=True)
else:
    print('Training disabled. No model has been started by this cell.')
subprocess.run([sys.executable, '-u', 'scripts/kaggle_lastfm.py', 'status'], check=True)


## Báo cáo và giữ checkpoint

Bật `RUN_REPORT = True` để tạo bảng từ kết quả đã ghi. ZIP kết quả lưu registry, toàn bộ checkpoint/history/summary và kết quả theo user. Lưu phiên bản Kaggle kèm output trước khi kết thúc; phiên sau thêm output đó vào Input. Gói chỉ tiếp tục lượt đã hoàn tất, không phục hồi optimizer giữa epoch.

**Không chạy lại từ đầu trên một output đã có test scores.** Giữ cùng payload source/data và dùng chức năng tiếp tục. File `.git` của phiên cũ không cần khôi phục vì code được clone lại từ bundle.

In [ ]:
if RUN_REPORT:
    subprocess.run([sys.executable, '-u', 'scripts/kaggle_lastfm.py', 'report'], check=True)
subprocess.run([sys.executable, '-u', 'scripts/kaggle_lastfm.py', 'checkpoint'], check=True)
print('Save this file from Kaggle Output: /kaggle/working/tkgat-lastfm-results.zip')
print('Reports, when requested:', PROJECT_ROOT / 'reports')
